# 🧬 Notebook Guide — AlphaFold 3 Evaluation & Confidence

## Learning Objectives
By the end of this notebook you will be able to:
- [ ] Compute PAE (Predicted Aligned Error) matrix numerically
- [ ] Calculate pTM and ipTM scores from a PAE matrix
- [ ] Implement DockQ from scratch (protein-protein docking quality)
- [ ] Compute interface RMSD and ligand RMSD
- [ ] Understand SE(3)-equivariant features and why AF3 needs them
- [ ] Build the OpenProteinSet data pipeline understanding
- [ ] Interpret AF3 confidence output: when to trust a prediction

## Why This Matters
Evaluation metrics are what separates a researcher who uses AF3 from one who builds it:
- **PAE** tells you which domain-domain orientations AF3 is confident about
- **ipTM** is the primary metric for protein-protein complex accuracy
- **DockQ** is what computational biology ML teams/structural biology research labs use to benchmark AF3 vs competitors
- **SE(3) equivariance** is why you cannot just add raw coordinates as features

---

## 🤖 Claude Integration — Copy These Prompts

**For PAE:**
```
Explain Predicted Aligned Error (PAE) in AlphaFold 2/3.
- PAE[i,j] = expected position error of residue j when residue i's frame is used
- How is PAE different from pLDDT? (pLDDT is per-residue; PAE is pairwise)
- What does a PAE matrix look like for a 2-domain protein? (low off-diagonal = domains move together)
- How do I use PAE to define protein domains? (connected components with PAE < 5Å)
- What PAE threshold indicates a reliable prediction? (PAE < 5Å is generally reliable)
```

**For ipTM/pTM:**
```
Explain pTM and ipTM in AlphaFold-Multimer and AF3.
- pTM: global TM-score estimate from the full PAE matrix
- ipTM: interface TM-score — only considers interface residues
- How is ipTM combined with pTM for ranking? AF3 ranking = 0.8*ipTM + 0.2*pTM
- What ipTM threshold indicates a reliable protein-protein complex? (ipTM > 0.75)
- Why is ipTM better than pTM for complexes? (global structure can be right but interface wrong)
```

**For DockQ:**
```
Explain DockQ scoring for protein-protein docking.
- DockQ combines 3 components: Fnat, LRMSD, IRMSD
- Fnat: fraction of native contacts preserved (contact = Cβ-Cβ < 8Å)
- LRMSD: RMSD of the ligand chain after superimposing receptor chain
- IRMSD: RMSD of interface residues after superimposing interface only
- DockQ = 0.49*Fnat + 0.53*(1/(1+(IRMSD/1.5)^2)) + 0.53*(1/(1+(LRMSD/8.5)^2))
- Categories: incorrect (<0.23), acceptable (0.23-0.49), medium (0.49-0.80), high (>0.80)
```

**For SE(3) Equivariance:**
```
Explain SE(3) equivariance and why it matters for protein structure.
- SE(3) = 3D rotations + translations (the symmetry group of 3D space)
- An equivariant model: rotating input → rotating output by same amount
- An invariant model: rotating input → same output (scalar quantities like energy)
- Why does AF3 need equivariant features? (protein structure is invariant to global rotation)
- Examples of invariant features: pairwise distances, angles between atoms
- Examples of equivariant features: coordinate vectors, normal vectors
- How does AF3 achieve this? Uses invariant pair features + equivariant coordinate updates
```

---

## Confidence Score Interpretation Guide
```
AF3 CONFIDENCE METRICS:

pLDDT (per-residue, 0-100):
  > 90: Very high confidence (dark blue) — backbone likely < 1Å RMSD
  70-90: High confidence (light blue)   — backbone likely < 2Å RMSD  
  50-70: Low confidence (yellow)        — mostly correct topology
  < 50: Very low confidence (orange)    — disordered region, do not trust

pTM (0-1, whole complex):
  > 0.8: High global confidence
  0.5-0.8: Moderate confidence
  < 0.5: Low confidence

ipTM (0-1, interface only):
  > 0.75: Reliable complex prediction  
  0.5-0.75: Uncertain — experimental validation needed
  < 0.5: Do not trust the interface

AF3 RANKING SCORE = 0.8 * ipTM + 0.2 * pTM
(For homo-oligomers or single chains: pTM only)

PAE (Predicted Aligned Error, in Å):
  < 5Å: Residues i and j are reliably positioned relative to each other
  5-15Å: Uncertain relative positioning
  > 15Å: No reliable information about relative position
```


## TL;DR — Plain English

**What this notebook does:** Teaches you to read, interpret, and critically evaluate AlphaFold3's output files — turning confidence scores into actionable biology.

**After this notebook you can:**
- Read a pLDDT score and know which parts of a predicted structure to trust
- Interpret a PAE (predicted aligned error) matrix to find domain boundaries and rigid units
- Calculate GDT-TS and TM-score to compare AF3 predictions against experimental structures
- Decide when an AF3 prediction is reliable enough to act on for drug design

**Why this matters:**
- HackerRank: Evaluation metric questions (what does pLDDT mean? how do you validate a structure prediction?) are standard in bioinformatics ML interviews
- computational biology ML teams: Every scientist at computational biology ML teams uses AF3 output daily; being able to critically assess confidence scores rather than blindly trust them is a key job skill

**Time:** ~2 hours | **Difficulty:** ⭐⭐⭐ | **Prerequisites:** 07/01 (AF3 architecture), 03/01 (structure analysis)

## Beginner Teaching Frame

**Who should fully work through this notebook:** advanced students. Beginners should treat this notebook as a conceptual first pass.

**How to study it on a first pass:**
- aim to explain the big idea of each component
- do not get stuck on every tensor shape immediately
- learn what the block is trying to accomplish biologically and geometrically

**Common traps:** drowning in implementation detail before understanding the role of Pairformer, FAPE, PAE, or diffusion in the full system.


## Canonical Resource Upgrade

Use these in order:
- [OpenFold](https://github.com/aqlaboratory/openfold) for code orientation
- [AlphaFold 2 paper](https://www.nature.com/articles/s41586-021-03819-2)
- [AlphaFold 3 paper](https://www.nature.com/articles/s41586-024-07487-w)


## 🗺️ Prerequisites & Learning Path

| | |
|--|--|
| 🔑 Prerequisites | 07/02 (AF3 data pipeline — need parsed structures to evaluate), 03/01 (RMSD basics) |
| 📍 You Are Here | Module 07/03 — AF3 Evaluation Metrics |
| ➡️ Next Steps | 07/04 (full-scale training) |
| 🏁 Goal | Implement TM-score and lDDT from scratch; correctly apply DockQ, PAE, and pLDDT in context |

### 🆕 From Scratch? Start Here:
1. [TM-score paper (Zhang 2004, PMC1464906)](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC1464906/) — 3 pages, read first
2. 03/01 (this repo) — RMSD and basic structural comparison
3. 07/02 (this repo) — how structures are parsed before evaluation
4. This notebook — evaluation metrics
**Time:** 8–12h | **Difficulty:** ⭐⭐⭐⭐

### 🔗 Cross-References
- Builds on: 07/02 (parsed structure objects), 03/01 (RMSD as baseline comparison metric)
- Used by: 07/04 (full-scale training uses these metrics for checkpointing), 10/01 (capstone evaluation)
- Parallel: 03/01 for RMSD basics (simpler metric, same structural alignment context)

## What This Notebook Teaches (Plain English)

**After AF3 predicts a structure, how do you know if the prediction is correct?**

This notebook covers the metrics used to evaluate and rank AF3 predictions — both the automatic confidence scores AF3 outputs and the external validation metrics used in competitions like CASP.

### Required Background
- ✅ AF3 architecture basics (Notebook 07/01)
- ✅ Protein structure concepts (Notebook 03/01)
- ✅ Basic PyTorch

### Metrics Dictionary

| Metric | What it measures | Range | Good value |
|--------|-----------------|-------|-----------|
| **PAE** | Expected position error of residue j when aligned using residue i's frame | 0-32 Å | < 5 Å |
| **pTM** | Predicted TM-score — overall fold confidence | 0-1 | > 0.5 |
| **ipTM** | Interface pTM — confidence specifically at the contact region between two chains | 0-1 | > 0.6 |
| **AF3 ranking score** | 0.8×ipTM + 0.2×pTM — used to pick the best of 5 predictions | 0-1 | > 0.6 |
| **DockQ** | External validation: how good is a protein-protein docking prediction | 0-1 | > 0.49 (medium quality) |
| **lDDT** | Local distance difference test — fraction of inter-atom distances preserved | 0-1 | > 0.7 |

### Why Multiple Metrics?

- **RMSD** is simple but penalizes large proteins unfairly (bigger protein = bigger RMSD even if fold is correct).
- **TM-score** fixes this but ignores local accuracy.
- **PAE** gives a per-residue-pair picture of where AF3 is uncertain.
- **DockQ** specifically measures binding interface quality (crucial for drug design).
- **lDDT-PLI** specifically measures how well ligand contacts with the protein are reproduced.

Together, they give a complete picture of prediction quality.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# PAE (Predicted Aligned Error) matrix
def compute_pae(pred_coords, confidence_model_output):
    """PAE_ij = expected error in position of residue j when frame i is aligned.
    In practice, AF3 outputs PAE directly as a (L,L) matrix.
    Here we simulate it from coordinate uncertainty.
    """
    L = len(pred_coords)
    # Simulate: PAE high between flexible/disordered domains
    pae = torch.zeros(L, L)
    # Intra-domain: low PAE (< 2Å)
    domain1 = range(0, L//2)
    domain2 = range(L//2, L)
    for i in domain1:
        for j in domain1:
            pae[i,j] = 0.5 + torch.rand(1)*1.5
    for i in domain2:
        for j in domain2:
            pae[i,j] = 0.8 + torch.rand(1)*1.5
    # Inter-domain: high PAE (> 10Å for flexible linker)
    for i in domain1:
        for j in domain2:
            pae[i,j] = 10 + torch.rand(1)*5
            pae[j,i] = 10 + torch.rand(1)*5
    return pae

torch.manual_seed(42)
L = 40
pred_coords = torch.randn(L, 3) * 10
pae = compute_pae(pred_coords, None)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(pae.numpy(), cmap='Greens_r', vmin=0, vmax=20)
plt.colorbar(im, ax=ax, label='PAE (Å)')
ax.set_title('Predicted Aligned Error Matrix')
ax.axhline(L//2, color='red', lw=0.5, linestyle='--')
ax.axvline(L//2, color='red', lw=0.5, linestyle='--')
ax.set_xlabel('Residue j')
ax.set_ylabel('Residue i')
plt.tight_layout()
plt.savefig('/tmp/pae_matrix.png', dpi=72)
print(f"Mean intra-domain PAE: {pae[:L//2,:L//2].mean():.1f} Å")
print(f"Mean inter-domain PAE: {pae[:L//2,L//2:].mean():.1f} Å")
print("Low PAE = confident relative positioning between those residues")

## PAE — Predicted Aligned Error
### NxN confidence matrix for domain-domain orientations

In [ ]:
import torch
import numpy as np

def pae_to_ptm(pae_matrix, d0_scale=1.24):
    """Estimate pTM from PAE matrix (AF2 approach).

    pTM = max_interface (1/L * sum_ij 1/(1 + (PAE_ij/d0)^2))
    where d0 = 1.24 * (L-15)^(1/3) - 1.8
    """
    L = pae_matrix.shape[0]
    d0 = max(d0_scale * (L - 15)**(1/3) - 1.8, 0.5)

    tm_sum = (1 / (1 + (pae_matrix / d0)**2)).mean()
    return tm_sum.item(), d0

def iptm_from_pae(pae, chain1_len, chain2_len):
    """ipTM: TM-score estimate for interface residue pairs only."""
    L = chain1_len + chain2_len
    d0 = max(1.24 * (L - 15)**(1/3) - 1.8, 0.5)
    # Interface = pairs where one residue is in chain1 and other in chain2
    interface_pae = pae[:chain1_len, chain1_len:]
    iptm = (1 / (1 + (interface_pae / d0)**2)).mean()
    return iptm.item()

torch.manual_seed(42)
L = 100
# Good complex: low PAE overall
pae_good = torch.rand(L, L) * 3   # 0-3 Å
pae_bad  = torch.rand(L, L) * 15  # 0-15 Å (uncertain)

ptm_good, d0 = pae_to_ptm(pae_good)
ptm_bad,  _  = pae_to_ptm(pae_bad)
print(f"d0 for L={L}: {d0:.2f} Å")
print(f"pTM (good structure): {ptm_good:.3f}")
print(f"pTM (bad structure):  {ptm_bad:.3f}")
print()
# ipTM for a 50+50 complex
iptm_good = iptm_from_pae(pae_good, 50, 50)
iptm_bad  = iptm_from_pae(pae_bad, 50, 50)
print(f"ipTM (good interface): {iptm_good:.3f}")
print(f"ipTM (bad interface):  {iptm_bad:.3f}")
print()
print("AF3 ranking score: 0.8*ipTM + 0.2*pTM")
print(f"  good: {0.8*iptm_good + 0.2*ptm_good:.3f}")

## pTM and ipTM Scores
### TM-score estimates from PAE matrix

In [ ]:
import numpy as np

def dockq(pred_ca, true_ca, pred_interface, true_interface,
          native_contacts, pred_contacts,
          ca_rmsd, i_rmsd_threshold=10.0):
    """
    DockQ score — composite metric for protein-protein docking quality.
    Components:
      fnat  = fraction of native contacts correctly predicted
      lrms  = RMSD of ligand CA after superimposing receptor
      irms  = interface RMSD
    DockQ = (fnat + 1/(1+(lrms/8.5)^2) + 1/(1+(irms/1.5)^2)) / 3
    """
    fnat = len(set(pred_contacts) & set(native_contacts)) / max(len(native_contacts), 1)
    lrms_score = 1 / (1 + (ca_rmsd / 8.5)**2)
    irms_score = 1 / (1 + (ca_rmsd / 1.5)**2)
    dockq_score = (fnat + lrms_score + irms_score) / 3
    return dockq_score, fnat, lrms_score, irms_score

# Simulate
np.random.seed(42)
native_contacts = [(i, j) for i in range(40, 60) for j in range(60, 80)
                   if np.random.rand() > 0.7]
print(f"Native contacts: {len(native_contacts)}")

for rmsd, n_correct in [(0.5, len(native_contacts)),
                         (2.0, int(len(native_contacts)*0.8)),
                         (8.0, int(len(native_contacts)*0.3))]:
    pred_contacts = native_contacts[:n_correct] + [(100+i,200+i) for i in range(5)]
    dq, fnat, lrms_s, irms_s = dockq(None, None, None, None,
                                       native_contacts, pred_contacts, rmsd)
    print(f"RMSD={rmsd:4.1f}Å fnat={fnat:.2f} DockQ={dq:.3f}  "
          f"→ {'High' if dq>0.6 else 'Medium' if dq>0.23 else 'Low'} quality")

## DockQ — Protein Docking Quality
### Gold standard evaluation metric used in AF3 benchmarks

In [ ]:
import torch
import numpy as np

def check_equivariance(model_fn, x, R=None, t=None):
    """Verify SE(3) equivariance: f(Rx+t) = R*f(x)+t (for coordinates).
    Or invariance: f(Rx+t) = f(x) (for scalars like energy/score).
    """
    if R is None:
        # Random rotation matrix
        theta = torch.tensor(np.pi / 4)
        R = torch.tensor([[torch.cos(theta), -torch.sin(theta), 0],
                          [torch.sin(theta),  torch.cos(theta), 0],
                          [0,                  0,               1]], dtype=torch.float)
    if t is None:
        t = torch.tensor([1.0, 2.0, 3.0])

    # Compute f(x) and f(Rx+t)
    out_x = model_fn(x)
    x_transformed = x @ R.T + t
    out_transformed = model_fn(x_transformed)

    # For invariant function: out should be same
    max_diff = (out_x - out_transformed).abs().max().item()
    return max_diff

# Invariant function: pairwise distances (should be SE(3)-invariant)
def pairwise_distances(x):
    return torch.cdist(x, x)

# Non-invariant function: coordinate mean (changes with rotation)
def coord_mean(x):
    return x.mean(dim=0)

torch.manual_seed(42)
x = torch.randn(10, 3)
diff_invariant = check_equivariance(pairwise_distances, x)
diff_noninvariant = check_equivariance(coord_mean, x)

print(f"Pairwise distances (invariant): max diff = {diff_invariant:.8f} (should ≈ 0)")
print(f"Coordinate mean (NOT invariant): max diff = {diff_noninvariant:.4f}")
print()
print("AF3 key: the diffusion module outputs SE(3)-equivariant coordinates")
print("Pairformer pair features are SE(3)-invariant (from distances/orientations)")
print("Triangle attention is equivariant w.r.t. residue permutations")

## SE(3) Equivariance
### Why AF3 features must be rotation/translation invariant

In [ ]:
# AlphaFold3 Evaluation — Interview Q&A
print("=" * 60)
print("AF3 EVALUATION — INTERVIEW Q&A")
print("=" * 60)

qas = [
    ("Q: What is PAE and how is it different from pLDDT?",
     "A: pLDDT (per-residue) measures local accuracy of a single residue's conformation "
     "(0-100, higher=better). PAE (N×N matrix) measures the expected error in residue j's "
     "position when residue i is used as the alignment frame — it captures RELATIVE domain "
     "orientations. Low PAE between two domains means AF3 is confident they are correctly "
     "positioned relative to each other."),

    ("Q: What threshold separates a 'good' DockQ score?",
     "A: DockQ > 0.8 = high quality, 0.6-0.8 = medium, 0.23-0.6 = acceptable, < 0.23 = incorrect. "
     "It combines fnat (fraction of native contacts), L-RMSD (ligand RMSD), and I-RMSD (interface RMSD)."),

    ("Q: How does ipTM differ from pTM?",
     "A: pTM estimates a TM-score for the full complex. ipTM estimates the TM-score "
     "for interface residue pairs only (cross-chain PAE). AF3 uses 0.8*ipTM + 0.2*pTM as "
     "the primary ranking metric for complex structures."),

    ("Q: Why is lDDT preferred over RMSD for local accuracy?",
     "A: RMSD requires superimposition (Kabsch algorithm) and is dominated by outlier residues. "
     "lDDT measures the fraction of pairwise distances preserved within thresholds (0.5/1/2/4Å), "
     "is superimposition-free, and is robust to domain movements."),
]

for q, a in qas:
    print(f"\n{q}")
    print(f"  {a}")

## Interview Q&A — AF3 Evaluation

In [ ]:
# Module 07/03 Resources
print("Key evaluation metrics and tools:")
print()
refs = [
    ("lDDT paper", "Mariani et al. 2013 Bioinformatics"),
    ("DockQ paper", "Basu & Wallner 2016 PLOS ONE"),
    ("TM-score paper", "Zhang & Skolnick 2004"),
    ("PAE in AF2", "Jumper et al. 2021 Supplementary"),
    ("CASP evaluation", "casp.predictioncenter.org"),
    ("OpenStructure", "openstructure.org — lDDT/TM-score/DockQ library"),
]
for name, ref in refs:
    print(f"  {name}: {ref}")
print()
print("Key thresholds (memorize for interviews):")
thresholds = {
    "TM-score": "> 0.5 = same fold, > 0.7 = good, ~1.0 = perfect",
    "RMSD":     "< 1Å = excellent, < 2Å = good, > 5Å = poor",
    "lDDT":     "> 0.9 = excellent, > 0.7 = good, > 0.5 = acceptable",
    "DockQ":    "> 0.8 = high, > 0.6 = medium, > 0.23 = acceptable",
    "pLDDT":    "> 90 = very confident, > 70 = confident, < 50 = disordered",
    "PAE":      "< 5Å = confident relative position",
}
for metric, rule in thresholds.items():
    print(f"  {metric}: {rule}")

## 📚 Resources

### 1️⃣ Theory — Foundations & Math
- **TM-score derivation**: TM = (1/L_target) Σ 1/(1 + (dᵢ/d₀)²) where d₀ = 1.24(L−15)^(1/3) − 1.8 — length-normalized
- **lDDT**: local Distance Difference Test — 4 distance thresholds (0.5, 1, 2, 4Å), no superposition needed, per-residue score
- **RMSD limitations**: sensitive to outliers, not length-normalized, requires global superposition — misleading for flexible proteins
- **DockQ**: composite docking quality score combining Fnat (native contacts), LRMS, iRMS — range [0,1]
- **Bootstrap statistics**: resampling-based significance testing for structure comparison metrics

### 2️⃣ Must-Have Popular Resources
- 📘 TM-score paper (Zhang & Skolnick, PMC1464906, 2004) — original derivation; 3 pages, essential reading
- 📘 lDDT paper (Mariani 2013, Bioinformatics) — defines the 4-threshold local metric used in AF pLDDT
- 📘 DockQ paper (Basu & Wallner 2016, PLoS ONE) — standard docking evaluation metric
- 📘 CASP15 assessment papers — blind prediction benchmarks; current state-of-the-art reference
- 🎓 CASP15 results presentation (YouTube) — visual summary of structure prediction progress
- ⭐ GitHub [bjornwallner/DockQ](https://github.com/bjornwallner/DockQ) — official DockQ implementation
- ⭐ GitHub [steineggerlab/foldseek](https://github.com/steineggerlab/foldseek) — ultra-fast structural search (3D-alignment)

### 3️⃣ Practicals — Hands-On
- 💻 Implement TM-score in NumPy — Kabsch superposition + per-residue scoring, validate against TM-align server
- 💻 Compute lDDT matrix for a predicted vs experimental structure — visualize per-residue confidence
- 💻 Parse PAE (Predicted Aligned Error) JSON from AF server — plot contact probability map
- 🤗 HuggingFace Space: [ESMFold](https://huggingface.co/facebook/esmfold_v1) — predict structure then evaluate with your metrics
- 📊 Kaggle: [Leash BELKA protein-ligand binding](https://www.kaggle.com/competitions/leash-BELKA) — binding evaluation challenge

### 4️⃣ Real-World Problems
- 🏭 Schrödinger / Vividion: DockQ used in production for docking pose validation pipelines
- 🏭 CAMEO blind evaluation server — continuous automated model evaluation, real-time AF3 monitoring
- 📊 Dataset: [CASP15 targets](https://predictioncenter.org/casp15/) — experimental structures + predictions from all groups
- 🔬 Application: Ensemble uncertainty quantification — multiple AF3 seeds → spread of TM-scores → confidence interval

### 5️⃣ Interview Tips
- ❓ Must know: When to use TM-score vs RMSD — TM-score when comparing proteins of different lengths; RMSD only for same structure comparison
- ❓ Must know: DockQ thresholds — >0.8 high quality, 0.49–0.8 medium, 0.23–0.49 acceptable, <0.23 incorrect
- ❓ Must know: pLDDT ≠ accuracy — it's a predicted confidence score, not a guaranteed accuracy measure
- ⚠️ Common mistake: Using RMSD alone for flexible multi-domain proteins — a hinge motion gives high RMSD despite correct structure
- 💡 Pro tip: Always report lDDT for local quality and TM-score for global fold quality — they are complementary, not redundant

### 6️⃣ Hot Industry Topics
- 🔥 Trending: [AF3 lDDT-PLI](https://alphafold.ebi.ac.uk/) — interface-specific lDDT metric for protein–ligand contacts
- 🔥 Trending: [steineggerlab/foldseek](https://github.com/steineggerlab/foldseek) — structural search over 200M AlphaFold structures in seconds
- 🔥 Trending: [ProteinBenchmark] — unified evaluation framework for structure prediction across all metrics
- 🚀 Build this: An evaluation dashboard that takes two PDB files, computes TM-score + lDDT + DockQ, and renders a per-residue quality plot

### Time Complexity — Protein Structure Evaluation Metrics
| Metric | Time | Notes |
|--------|------|-------|
| RMSD (n atoms) | O(n) after alignment | O(n) for Kabsch |
| TM-score | O(n²) | pairwise distances + optimization |
| lDDT (n atoms) | O(n²) | all pairwise within 15Å |
| pTM / ipTM | O(n²) | from PAE matrix |
| DockQ | O(n_interface²) | interface atoms only |
| GDT_TS | O(n²) | distance thresholds |
| GDT_HA | O(n²) | stricter thresholds |
| PAE (pairwise alignment error) | O(n²) | n = residues |
| Contact precision@L/5 | O(n²) | top L/5 predicted contacts |

In [ ]:
# Module 07/03 extended resources
print("Extended resources for AF3 evaluation:")
print("  * ModelAngelo: automated model building from cryo-EM + AF3")
print("  * MolProbity: Ramachandran, clashscore, rotamer quality")
print("  * DALI server: structural comparison with PDB library")
print("  * US-align: universal structure alignment (protein/RNA/DNA)")
print()
print("CASP15 (2022) results:")
print("  - AF2 variants dominated; ESMFold competitive on single-chain")
print("  - Complex prediction still challenging → motivation for AF3")
print()
print("AlphaFold-Multimer vs AlphaFold3 for complexes:")
print("  - AF-Multimer: extended AF2 with cross-chain templates")
print("  - AF3: unified diffusion model + ligand support + DNA/RNA")

# 🌍 Real World Problems — AF3 Evaluation

## Problem 1 — Reproduce AF3 DockQ on CASP15 Targets
**Dataset/Source**: CASP15 server predictions (free download) + PDB experimental structures
**GitHub**: [github.com/bjornwallner/DockQ](https://github.com/bjornwallner/DockQ) — official DockQ implementation
**Hugging Face**: [huggingface.co/datasets/chandar-lab/ProteinBench](https://huggingface.co/datasets/chandar-lab/ProteinBench)
**Skills**: DockQ, Fnat, iRMSD, LRMSD, docking benchmark analysis

```python
import urllib.request

def fetch_and_evaluate_casp15(target_id, pred_pdb_url, true_pdb_id):
    """
    Evaluate an AF3 prediction against the experimental structure.
    target_id: CASP15 target ID (e.g. 'H1171')
    pred_pdb_url: URL to AF3's predicted PDB file
    true_pdb_id: PDB ID of experimental structure
    """
    # Fetch structures
    # true_pdb = fetch_pdb(true_pdb_id)  # from RCSB
    # Extract chain coordinates
    # n_true, ca_true, c_true = extract_backbone(true_pdb, 'A')
    # pred_coords = ... (parse prediction file)
    pass

# YOUR TASK: Download CASP15 predictions from predictioncenter.org/casp15
# 1. Pick 5 protein-protein targets (H-category)
# 2. Download AF3 server predictions (group 'MULTICOM_AF3')
# 3. Download experimental structures from PDB
# 4. Run dockq_score() on each
# 5. Reproduce the ~67% success rate (DockQ > 0.23) reported in the AF3 paper

# Quick sanity check with perfect prediction
true_A = np.random.randn(20, 3) * 5
true_B = true_A[:5] + np.random.randn(10, 3) * 2 + np.array([8, 0, 0])
pred_A_perfect = true_A.copy()
pred_B_perfect = true_B.copy()
dq, fn, ir, lr, cat = dockq_score(pred_A_perfect, pred_B_perfect, true_A, true_B)
print(f'Perfect prediction: DockQ={dq:.3f}, Fnat={fn:.3f}, category={cat}')
# DockQ should be ~1.0 for perfect prediction
```

**Real impact**: AF3's DockQ improvement over AF2-Multimer is the primary evidence in the Nature 2024 paper — understanding how to compute DockQ is essential for evaluating structure prediction models.

---

## Problem 2 — Build a PAE Viewer to Detect Flexible Linkers
**Dataset/Source**: AF3 predictions for calmodulin (2 EF-hand domains connected by linker)
**GitHub**: [github.com/google-deepmind/alphafold](https://github.com/google-deepmind/alphafold/blob/main/notebooks/AlphaFold.ipynb)
**Hugging Face**: [huggingface.co/spaces/simonduerr/esmfold](https://huggingface.co/spaces/simonduerr/esmfold)
**Skills**: PAE computation, domain detection, flexible linker identification

```python
# Calmodulin: 148 residues, 2 EF-hand domains (1-75 and 80-148)
# Connected by a flexible helix/linker at residues 65-82
# PAE should show: two blue blocks on diagonal, white/yellow inter-domain block

def visualize_pae_text(pae_matrix, block_size=5):
    """ASCII visualization of PAE matrix (for terminal display)."""
    N = pae_matrix.shape[0]
    max_pae = pae_matrix.max()
    print(f'PAE Matrix ({N}x{N}), max={max_pae:.1f}Å')
    print('  (dark=low PAE=confident, light=high PAE=uncertain)')
    
    for i in range(0, N, block_size):
        row_str = ''
        for j in range(0, N, block_size):
            block = pae_matrix[i:i+block_size, j:j+block_size]
            avg = block.mean()
            if avg < 3:   row_str += '██'
            elif avg < 8:  row_str += '▓▓'
            elif avg < 15: row_str += '░░'
            else:          row_str += '  '
        print(row_str)

# Simulate calmodulin PAE
N_cal = 30  # simplified 30-residue version (3 blocks of 10)
cal_pae = np.zeros((N_cal, N_cal))
# Domain 1-1 (residues 0-9): low PAE
cal_pae[:10, :10] = np.random.exponential(2, (10, 10))
# Domain 2-2 (residues 20-29): low PAE  
cal_pae[20:, 20:] = np.random.exponential(2, (10, 10))
# Inter-domain (0-9 vs 20-29): high PAE
cal_pae[:10, 20:] = np.random.uniform(10, 20, (10, 10))
cal_pae[20:, :10] = cal_pae[:10, 20:].T
# Linker region (10-19): moderate PAE
cal_pae[10:20, :] = np.random.uniform(5, 15, (10, N_cal))
cal_pae[:, 10:20] = cal_pae[10:20, :].T

visualize_pae_text(cal_pae, block_size=5)
domains = pae_to_domains(cal_pae, threshold=5.0)
print(f'Detected {len(domains)} domains:')
for i, d in enumerate(sorted(domains, key=len, reverse=True)):
    print(f'  Domain {i+1}: residues {[x+1 for x in d]}')

# YOUR TASK: Download AF3 JSON result from alphafold.ebi.ac.uk
# JSON file contains 'predicted_aligned_error' matrix
# import json; pae = json.load(open('AF3_result.json'))['predicted_aligned_error']
```

**Real impact**: The PAE matrix from AF3 is used by structural biologists to decide whether domain-domain orientations in a predicted complex are reliable — directly impacts which drug targets to pursue.

---

## Problem 3 — Compute SE(3)-Invariant Features for Protein Graph
**GitHub**: [github.com/drorlab/e3nn](https://github.com/e3nn/e3nn) — equivariant neural networks library
**Hugging Face**: [huggingface.co/datasets/nferruz/ProteinGym](https://huggingface.co/datasets/nferruz/ProteinGym)
**Kaggle**: [CAFA5 Protein Function](https://www.kaggle.com/competitions/cafa-5-protein-function-prediction)
**Skills**: SE(3) invariance, protein graph construction, equivariant features

```python
def build_invariant_protein_graph(ca_coords, aa_types, k_nearest=16):
    """Build SE(3)-invariant protein graph for GNN input.
    
    Nodes: residues with invariant features (aa type, depth in structure)
    Edges: k-nearest neighbor Cα contacts with invariant edge features
    """
    N = len(ca_coords)
    
    # Node features (all invariant)
    node_features = []
    for i in range(N):
        aa_onehot = np.zeros(21)
        aa_onehot[aa_types[i]] = 1.0
        # Relative depth in chain (position / N, invariant to 3D rotation)
        position = np.array([i / N])
        node_features.append(np.concatenate([aa_onehot, position]))  # (22,)
    node_feat = np.array(node_features)  # (N, 22)
    
    # Edge features (pairwise invariant features)
    dist_matrix = np.sqrt(np.sum((ca_coords[:,None,:]-ca_coords[None,:,:])**2, axis=-1))
    
    edges = []
    edge_features = []
    for i in range(N):
        # k nearest neighbors (by Cα distance)
        neighbor_dists = dist_matrix[i]
        neighbor_dists[i] = np.inf  # exclude self
        k_near = np.argsort(neighbor_dists)[:k_nearest]
        
        for j in k_near:
            d = dist_matrix[i, j]
            seq_sep = abs(i - j)
            # RBF encoding of distance (invariant)
            rbf = np.exp(-(d - np.linspace(0, 20, 16))**2 / 4)
            # Sequence separation binning (invariant)
            sep_bin = min(int(np.log2(seq_sep + 1)), 7)  # log-binned
            sep_feat = np.zeros(8); sep_feat[sep_bin] = 1.0
            
            edges.append((i, j))
            edge_features.append(np.concatenate([rbf, sep_feat]))  # (24,)
    
    return node_feat, np.array(edges), np.array(edge_features)

# Test
np.random.seed(42)
N_prot = 20
coords = np.cumsum(np.random.randn(N_prot, 3) * 3.8, axis=0)  # protein-like walk
aa_types = np.random.randint(0, 20, N_prot)

node_feat, edges, edge_feat = build_invariant_protein_graph(coords, aa_types, k_nearest=8)

# Verify invariance
R = np.linalg.qr(np.random.randn(3,3))[0]
coords_rot = (R @ coords.T).T + np.random.randn(3)*10
nf_rot, e_rot, ef_rot = build_invariant_protein_graph(coords_rot, aa_types, k_nearest=8)
max_diff = np.max(np.abs(edge_feat - ef_rot))

print(f'SE(3)-Invariant Protein Graph:')
print(f'  Node features: {node_feat.shape}  (N, 22)')
print(f'  Edges: {edges.shape}  ({len(edges)} edges for {N_prot} residues, k=8 NN)')
print(f'  Edge features: {edge_feat.shape}  (E, 24)')
print(f'  Invariance check (edge features after rotation): max_diff={max_diff:.2e}')
print(f'  -> Features ARE SE(3)-invariant: {max_diff < 1e-10}')
```

**Real impact**: Invariant protein graphs are the input format for every modern protein GNN (ESM-GVP, EquiFold, AF3's pair representation) — this is the foundational data structure of computational drug discovery.

---

## Resources
| Resource | URL | Purpose |
|----------|-----|---------|
| DockQ Official | github.com/bjornwallner/DockQ | Gold standard docking metric |
| AF3 Paper | doi.org/10.1038/s41586-024-07487-w | Abramson et al. 2024 |
| CASP15 Predictions | predictioncenter.org/casp15 | Download AF3 server predictions |
| e3nn Library | github.com/e3nn/e3nn | SE(3)-equivariant neural networks |
| AlphaFold EBI | alphafold.ebi.ac.uk | 200M+ AF2 predictions + PAE JSON |
| ProteinGym | huggingface.co/datasets/nferruz/ProteinGym | Protein fitness benchmarks |
| CAFA5 Kaggle | kaggle.com/competitions/cafa-5-protein-function-prediction | $57K protein function challenge |

## Mastery Check

On a first pass, success means you can:
1. explain what this notebook's component does in the larger AF3 pipeline
2. describe why geometry matters here
3. explain at least one confidence or training metric in words
4. decide whether you should revisit this notebook later for deeper implementation work


## 🐛 Debug Exercise — AF3 Output Evaluation

Find and fix the **3 bugs** in the structure confidence evaluation code below.

**Expected correct output:**
```
High pLDDT residues (confident): residues with pLDDT > 70
PAE[i,j] != PAE[j,i] for i != j  (PAE is asymmetric by definition)
TM-score normalized by target length L0, not aligned length N
```

<details>
<summary>Show answer</summary>

**Bug 1 — pLDDT interpretation reversed:** Higher pLDDT = MORE confident (not less).
pLDDT > 70 is confident, > 90 is very confident, < 50 is disordered.
Fix: `confident_mask = plddt > 70` (not `< 70`).

**Bug 2 — PAE treated as symmetric:** PAE[i,j] = error in position of residue i when
residue j is the reference frame. This is NOT symmetric. Taking `(PAE + PAE.T) / 2`
destroys directional information used to identify domain interfaces.
Fix: use the raw PAE matrix directly.

**Bug 3 — TM-score using N (aligned) instead of L0 (target):** The standard TM-score
normalizer is the length of the *target* structure (L0), which is fixed. Using N (number
of aligned pairs) makes the score depend on alignment coverage.
Fix: `L0 = len(target_coords)` and use `L0` in the denominator.
</details>


In [ ]:
# DEBUG EXERCISE — AF3 output evaluation, find and fix 3 bugs
import numpy as np

L = 100  # protein length

# Simulated pLDDT scores (0-100)
np.random.seed(7)
plddt = np.random.uniform(20, 95, size=L)

# BUG 1: pLDDT interpretation reversed — higher IS more confident
# This code marks HIGH-confidence residues as disordered
confident_mask = plddt < 70     # WRONG — should be plddt > 70
disordered_mask = plddt > 70    # WRONG — should be plddt < 50

print(f"'Confident' residues (WRONG — using < 70): {confident_mask.sum()}")
print(f"'Disordered' residues (WRONG — using > 70): {disordered_mask.sum()}")
print("Expected: confident = residues with HIGH pLDDT (>70), not low")

# Simulated PAE matrix (asymmetric by definition)
PAE = np.abs(np.random.randn(L, L)) * 5
PAE[np.diag_indices(L)] = 0.0
# Make it genuinely asymmetric
PAE[0, 1] = 3.0
PAE[1, 0] = 12.0

# BUG 2: symmetrizing PAE destroys directional domain interface information
PAE_sym = (PAE + PAE.T) / 2     # WRONG — PAE is NOT symmetric
print(f"\nPAE[0,1] raw: {PAE[0,1]:.2f}, PAE[1,0] raw: {PAE[1,0]:.2f}  (asymmetric)")
print(f"After wrong symmetrization: PAE_sym[0,1] = {PAE_sym[0,1]:.2f} = PAE_sym[1,0] = {PAE_sym[1,0]:.2f}")

# TM-score calculation
def tm_score_buggy(mobile, target, d0=5.0):
    N = len(mobile)           # number of aligned residues
    L0 = N                    # BUG 3: should be len(target) — the target/reference length
                              # when N < len(target) this inflates the score
    dists = np.sqrt(np.sum((mobile - target)**2, axis=1))
    return (1.0 / L0) * np.sum(1.0 / (1.0 + (dists / d0)**2))

mobile = np.random.randn(80, 3)   # 80 aligned residues
target = np.random.randn(100, 3)  # target has 100 residues

# Only using first 80 residues of target for alignment
score = tm_score_buggy(mobile, target[:80])
print(f"\nTM-score (buggy, L0=N=80): {score:.3f}")
print("Bug: L0 should be len(target)=100 to penalize incomplete coverage")
